# Exploring Axial Seamount: seafloor uplift and vent temperature

Run the cells from top to bottom (Shift+Enter). Each section asks one question.
Change the code and re-run to explore your own.

You need Python with `pandas`, `matplotlib` and `jupyterlab`
(`pip install pandas matplotlib jupyterlab`). Keep this notebook in the same
folder as `botpt/` and `trhph/`. For a no-code first look, open `explore.html`
in a browser.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def load(path):
    return pd.read_csv(path, parse_dates=["time"], index_col="time")

ccal = load("botpt/RS03CCAL-MJ03F-05-BOTPTA301_central_caldera_hourly.csv")
ecal = load("botpt/RS03ECAL-MJ03E-06-BOTPTA302_eastern_caldera_hourly.csv")
vent = load("trhph/RS03INT1-MJ03C-10-TRHPHA301_international_district_hourly.csv")
ccal.head()

## 1. What does a file look like?

Each row is one hour. `uplift_m` is how far the seafloor has moved up (+) or
down (−) since the record started, with tides already removed. See `README.md`
for every column.

In [ ]:
ax = ccal.uplift_m.plot(figsize=(12, 3), lw=0.5, title="Central Caldera, hourly")
ax.set_ylabel("uplift (m)");

## 2. Clean up and simplify

Drop the hours flagged as sensor glitches (`spike_flag == 1`), then average to
one value per day. Daily data is easier to plot and compare over 12 years.

In [ ]:
def daily(df):
    if "spike_flag" in df:
        df = df[df.spike_flag == 0]
    return df.resample("1D").mean()

ccal_d, ecal_d, vent_d = daily(ccal), daily(ecal), daily(vent)

ax = ccal_d.uplift_m.plot(figsize=(12, 3), label="Central Caldera")
ecal_d.uplift_m.plot(ax=ax, label="Eastern Caldera")
ax.set_ylabel("uplift (m)")
ax.legend();

## 3. The April 2015 eruption

Magma drained out of the reservoir and the caldera floor dropped. Zoom in to
see how fast, and compare the center of the caldera with its edge.

In [ ]:
window = slice("2015-04-15", "2015-05-31")
ax = ccal.loc[window, "uplift_m"].plot(figsize=(12, 3), label="Central Caldera")
ecal.loc[window, "uplift_m"].plot(ax=ax, label="Eastern Caldera")
ax.set_ylabel("uplift (m)")
ax.legend()

for name, df in [("Central", ccal), ("Eastern", ecal)]:
    drop = df.loc["2015-04-22", "uplift_m"].mean() - df.loc["2015-05-20", "uplift_m"].mean()
    print(f"{name} Caldera dropped {drop:.2f} m")

## 4. How fast is the volcano re-inflating?

Average each year, then take the change from one year to the next. The years
around 2015 are distorted by the eruption, and 2014 and 2026 are partial years.

In [ ]:
yearly = ccal_d.uplift_m.resample("YE").mean()
rate = yearly.diff() * 100  # m -> cm per year
rate.index = rate.index.year
ax = rate.plot.bar(figsize=(10, 3), title="Central Caldera inflation rate")
ax.set(xlabel="", ylabel="cm per year");

## 5. Removing the ocean: center minus edge

Every sensor also records a few cm of ocean signal (currents, weather) that has
nothing to do with the volcano. All sites feel it about equally, so subtracting
one site from another cancels most of it. What's left shows how the center of
the caldera moves *relative to* its edge.

In [ ]:
both = pd.concat([ccal_d.uplift_m.rename("center"), ecal_d.uplift_m.rename("edge")], axis=1).dropna()
diff = both.center - both.edge

fig, axs = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
both.loc["2022":, "center"].plot(ax=axs[0], title="Central Caldera alone")
diff.loc["2022":].plot(ax=axs[1], title="Central minus Eastern Caldera")
for ax in axs:
    ax.set_ylabel("m")

print(f"day-to-day scatter: center alone {both.center.diff().std() * 100:.1f} cm, "
      f"center minus edge {diff.diff().std() * 100:.1f} cm")

## 6. Seafloor uplift and vent temperature together

Does the hydrothermal vent respond to what the volcano is doing? Plot both on
one time axis. **Careful:** the temperature probe is re-placed in the vent at
every new deployment (gray dotted lines), so a jump at a dotted line may come
from the new placement, not from the vent.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4))
ccal_d.uplift_m.plot(ax=ax1, color="tab:blue")
ax1.set_ylabel("Central Caldera uplift (m)", color="tab:blue")

ax2 = ax1.twinx()
vent_d.temperature_mean_degC.plot(ax=ax2, color="tab:red", lw=0.8)
ax2.set_ylabel("vent temperature (°C)", color="tab:red")

dep = vent.deployment.dropna()
for t in dep.index[dep.diff() > 0]:
    ax1.axvline(t, color="gray", ls=":", lw=0.8)

## Ideas to explore next

- Load the ASHES and International District BOTPT files. How much does each
  site move compared with Central Caldera? What does that suggest about where
  the magma reservoir is?
- Compare Central Caldera's uplift today with its level just before the 2015
  eruption. Researchers have used this kind of comparison to think about when
  Axial might erupt next.
- Zoom into the hourly data on 2015-04-24. When exactly did deflation start?
- Pick one TRHPH deployment with steady temperatures and look for trends or
  daily cycles within it.